In [1]:
import torch
import torch.nn as nn
import random
import json
import argparse
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from collections import defaultdict
import torch.nn.functional as F
# from tqdm.auto import tqdm

from infer_batch_ipynb import id_mapping, feats, graph, labels, split_idx, model, data_loader

/root/miniconda3/envs/graphmae/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Use sce_loss and alpha_l=2 ===
num_encoder_params: 658434, num_decoder_params: 395520, num_params_in_total: 1842948


/workspace/baseline/GraphMae2/infer_batch_ipynb.py:32: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.



16


In [2]:
import time
def sample_and_expand_users(
    user_map_path=None,
    similar_path=None,
    dissimilar_path=None,
    seed_batch_size=10,
    k=5,
    random_seed=42,
    user_map=None,
    similar_users_dict=None,
    dissimilar_users_dict=None,
    min_similar=1,
    min_dissimilar=1,
    normalize_input=True,
    verbose=True
):
    """
    Optimized version to sample seed users and expand with similar/dissimilar users.
    Now avoids failures by validating sim/dissim overlap during filtering.
    """
    start_time = time.time()

    # --- 1) Load if not preloaded ---
    if user_map is None:
        with open(user_map_path, 'r') as f:
            user_map = json.load(f)
    if similar_users_dict is None:
        with open(similar_path, 'r') as f:
            similar_users_dict = json.load(f)
    if dissimilar_users_dict is None:
        with open(dissimilar_path, 'r') as f:
            dissimilar_users_dict = json.load(f)

    # --- 2) Normalize all user IDs to strings ---
    if normalize_input:
        user_map = {str(k): v for k, v in user_map.items()}
        similar_users_dict = {str(k): [str(n) for n in v] for k, v in similar_users_dict.items()}
        dissimilar_users_dict = {str(k): [str(n) for n in v] for k, v in dissimilar_users_dict.items()}

    valid_user_set = set(user_map.keys())
    random.seed(random_seed)

    # --- 3) Filter neighbors ---
    def filter_neighbors(neighbors_dict):
        return {
            user: [nbr for nbr in nbrs if nbr in valid_user_set and nbr != user]
            for user, nbrs in neighbors_dict.items()
            if user in valid_user_set
        }

    filtered_similar = filter_neighbors(similar_users_dict)
    filtered_dissimilar = filter_neighbors(dissimilar_users_dict)

    # --- 4) Check candidates AFTER dissimilar overlap removal ---
    candidate_users = set(filtered_similar.keys()) & set(filtered_dissimilar.keys())
    valid_users = []
    for user in candidate_users:
        sim = set(filtered_similar.get(user, []))
        dissim = set(filtered_dissimilar.get(user, [])) - sim
        if len(sim) >= min_similar and len(dissim) >= min_dissimilar:
            valid_users.append(user)

    if len(valid_users) < seed_batch_size:
        if verbose:
            print(
                f"Warning: Only {len(valid_users)} users have >= {min_similar} similar and >= {min_dissimilar} dissimilar after filtering overlap. "
                f"Adjusting seed_batch_size from {seed_batch_size} to {len(valid_users)}."
            )
        seed_batch_size = len(valid_users)

    if seed_batch_size == 0:
        raise ValueError("No users have enough similar/dissimilar neighbors. Check your data.")

    # --- 5) Sample seeds & prepare data ---
    U_b = random.sample(valid_users, seed_batch_size)
    U_b_similar = {}
    U_b_dissimilar = {}
    expanded_users = set(U_b)

    for user in U_b:
        sim_candidates = filtered_similar[user]
        dissim_candidates = [d for d in filtered_dissimilar[user] if d not in sim_candidates]

        sim_selected = random.sample(sim_candidates, min(k, len(sim_candidates)))
        dissim_selected = random.sample(dissim_candidates, min(k, len(dissim_candidates)))

        # ✅ No need for another min_similar/min_dissimilar check here anymore
        U_b_similar[user] = sim_selected
        U_b_dissimilar[user] = dissim_selected
        expanded_users.update(sim_selected)
        expanded_users.update(dissim_selected)

        if verbose and (len(sim_selected) < k or len(dissim_selected) < k):
            print(f"Note: User '{user}' has only {len(sim_selected)} similar and {len(dissim_selected)} dissimilar (requested k={k}).")

    U_b_expanded = list(expanded_users)

    if verbose:
        print(
            f"Sampled {len(U_b)} seed users. Expanded set size: {len(U_b_expanded)}. "
            f"Took {time.time() - start_time:.2f} seconds."
        )

    return U_b, U_b_expanded, U_b_similar, U_b_dissimilar

In [3]:
user_map_path = "/workspace/data/pretrain/twitter/user_map.json"
similar_path = "/workspace/data/pretrain/twitter/similar_users_rbf_full_90.json"
dissimilar_path = "/workspace/data/pretrain/twitter/dissimilar_users_rbf_full_90.json"
device = "cuda" if torch.cuda.is_available() else "cpu"

with open(user_map_path, "r") as f:
    user_map = json.load(f)
with open(similar_path, "r") as f:
    similar_users_dict = json.load(f)
with open(dissimilar_path, "r") as f:
    dissimilar_users_dict = json.load(f)

In [4]:
len(user_map.keys())

1506390

In [5]:
U_b, U_b_expanded, U_b_similar, U_b_dissimilar = sample_and_expand_users(
    user_map_path, similar_path, dissimilar_path,
    seed_batch_size=100, k=5, random_seed=42,
    user_map=user_map, similar_users_dict=similar_users_dict,
    dissimilar_users_dict=dissimilar_users_dict
)

Sampled 100 seed users. Expanded set size: 1100. Took 567.95 seconds.


In [29]:
#################################################
# Similarity and Dissimilary Loss
#################################################

def compute_sim_dissim_loss(
    embeddings,
    user_ids_batch,
    seed_users,
    U_b_similar,
    U_b_dissimilar,
    margin=0.2
):
    """
    Enforce that each seed user u is closer to users in U_b_similar[u]
    than to users in U_b_dissimilar[u], by at least 'margin':

    L_sd = average over all (u, v, w) of max(0, margin - (cos(u,v) - cos(u,w))).

    * embeddings:       Tensor [N, d]
    * user_ids_batch:   list of length N, giving the user_id for each row in 'embeddings'
    * seed_users:       the list U_b (seed)
    * U_b_similar[u]:   list of similar user IDs for user 'u'
    * U_b_dissimilar[u]: likewise dissimilar
    * margin:           margin for ranking
    """
    device = embeddings.device
    # Build quick lookup: user_id -> index in 'embeddings'
    user2idx = {}
    for i, uid in enumerate(user_ids_batch):
        user2idx[uid] = i

    total_cost = torch.tensor(0.0, device=device)
    triplet_count = 0

    for seed_u in seed_users:
        if seed_u not in user2idx:
            continue
        u_idx = user2idx[seed_u]
        z_u = embeddings[u_idx]  # shape [d]

        # all similar users for this seed
        sim_list = U_b_similar.get(seed_u, [])
        # all dissimilar
        dissim_list = U_b_dissimilar.get(seed_u, [])

        for sim_u in sim_list:
            if sim_u not in user2idx:
                continue
            z_sim = embeddings[user2idx[sim_u]]
            # cos(u, sim)
            cos_sim = F.cosine_similarity(
                z_u.unsqueeze(0), z_sim.unsqueeze(0), dim=-1
            )[0]

            for dis_u in dissim_list:
                if dis_u not in user2idx:
                    continue
                z_dis = embeddings[user2idx[dis_u]]
                # cos(u, dis)
                cos_dis = F.cosine_similarity(
                    z_u.unsqueeze(0), z_dis.unsqueeze(0), dim=-1
                )[0]

                # margin-based loss: want cos(u, sim) - cos(u, dis) >= margin
                cost = torch.relu(margin - (cos_sim - cos_dis))
                total_cost += cost
                triplet_count += 1

    if triplet_count == 0:
        return torch.tensor(0.0, device=device)

    return total_cost / triplet_count

import torch
import torch.nn as nn
import torch.nn.functional as F

def compute_cluster_loss_chunked(
    embeddings,
    cluster_labels,
    alpha=1.0,
    chunk_size=1024
):
    """
    Differentiable version of:

        L = mean_intra - alpha * mean_inter

    where:
      mean_intra = average distance among points in the same cluster (excluding self)
      mean_inter = average distance among points in different clusters

    We chunk over row & column blocks to avoid allocating a full NxN distance matrix at once,
    **but** we do NOT detach or move partial sums to CPU => the gradient flows fully.

    Args:
      embeddings:    (N, d) float tensor, requires_grad=True
      cluster_labels: length-N list/1D tensor of integer cluster IDs
      alpha:         float, weight factor for pushing different clusters apart
      chunk_size:    how many rows (and columns) to process at a time

    Returns:
      loss: scalar tensor, requires_grad=True
    """
    device = embeddings.device
    N = embeddings.size(0)
    if N == 0:
        # Return a zero that still requires grad
        return torch.tensor(0.0, device=device, requires_grad=True)

    # Convert cluster_labels to a tensor on the same device, if needed
    if not torch.is_tensor(cluster_labels):
        cluster_labels = torch.tensor(cluster_labels, dtype=torch.long, device=device)
    else:
        cluster_labels = cluster_labels.to(device)

    # We'll accumulate the total "same-cluster distance" and "diff-cluster distance"
    # as PyTorch Tensors that require grad. We initialize them to 0.
    same_dist_sum = torch.zeros([], device=device, requires_grad=True)
    diff_dist_sum = torch.zeros([], device=device, requires_grad=True)

    # The counts are just integer counters (no gradient needed),
    # so we keep them as regular Tensors with requires_grad=False.
    same_count = torch.zeros([], device=device, dtype=torch.long)
    diff_count = torch.zeros([], device=device, dtype=torch.long)

    # Iterate over row-blocks
    for i_start in range(0, N, chunk_size):
        i_end = min(N, i_start + chunk_size)
        block_i = embeddings[i_start:i_end]            # [chunk_i, d]
        block_labels_i = cluster_labels[i_start:i_end] # [chunk_i]

        # Compute pairwise distances with all columns in sub-blocks
        for j_start in range(0, N, chunk_size):
            j_end = min(N, j_start + chunk_size)
            block_j = embeddings[j_start:j_end]            # [chunk_j, d]
            block_labels_j = cluster_labels[j_start:j_end] # [chunk_j]

            # block_diff: [chunk_i, chunk_j, d]
            block_diff = block_i.unsqueeze(1) - block_j.unsqueeze(0)
            # block_dist: [chunk_i, chunk_j]
            block_dist = (block_diff * block_diff).sum(dim=-1)

            # same-cluster mask
            block_same_mask = (block_labels_i.unsqueeze(1) == block_labels_j.unsqueeze(0))
            
            # If this is the same block (i.e., i_start == j_start), exclude diagonal (i==j)
            if i_start == j_start:
                diag_len = block_same_mask.size(0)  # same as size(1) if chunk_i == chunk_j
                diag_mask = torch.eye(diag_len, dtype=torch.bool, device=device)
                block_same_mask = block_same_mask & ~diag_mask

            # different-cluster mask
            block_diff_mask = ~block_same_mask
            if i_start == j_start:
                # Exclude diagonal from diff_mask as well
                block_diff_mask = block_diff_mask & ~diag_mask

            # Sum distances where same_mask is True
            same_chunk_sum = block_dist[block_same_mask].sum()
            # Sum distances where diff_mask is True
            diff_chunk_sum = block_dist[block_diff_mask].sum()

            # Count how many pairs in each mask
            same_chunk_count = block_same_mask.sum()
            diff_chunk_count = block_diff_mask.sum()

            # Accumulate into the Tensors that require grad
            # We do NOT use in-place += on the requires_grad=True variables
            # because in-place can cause issues with autograd. Instead, reassign:
            same_dist_sum = same_dist_sum + same_chunk_sum
            diff_dist_sum = diff_dist_sum + diff_chunk_sum

            # Accumulate counts (no need for gradient)
            same_count = same_count + same_chunk_count
            diff_count = diff_count + diff_chunk_count

    # Now compute the mean distances
    # same_count and diff_count are just integer counts (grad=False).
    # same_dist_sum and diff_dist_sum have the computation graph for embedding-based distances.
    
    # If same_count is 0, that means no same-cluster pairs => mean_intra=0
    mean_intra = torch.where(
        same_count > 0,
        same_dist_sum / same_count.float().clamp_min(1),
        torch.zeros([], device=device, requires_grad=True)
    )

    # If diff_count is 0, that means no diff-cluster pairs => mean_inter=0
    mean_inter = torch.where(
        diff_count > 0,
        diff_dist_sum / diff_count.float().clamp_min(1),
        torch.zeros([], device=device, requires_grad=True)
    )

    # L = mean_intra - alpha * mean_inter
    loss = mean_intra - alpha * mean_inter
    return loss

def run_kmeans(embeddings, num_clusters=10):
    embeddings_np = embeddings.detach().cpu().numpy()
    kmeans = KMeans(n_clusters=num_clusters, random_state=42)
    kmeans.fit(embeddings_np)
    return kmeans.labels_

In [6]:
# Ensure similar/dissimilar users are mapped to nodes
expanded_node_indices = []
for u in U_b_expanded:
    val = user_map.get(u, [])
    if isinstance(val, list):
        expanded_node_indices.extend(val)
    elif val:
        expanded_node_indices.append(val)
for sim_list in U_b_similar.values():
    for u in sim_list:
        val = user_map.get(u, [])
        if isinstance(val, list):
            expanded_node_indices.extend(val)
        elif val:
            expanded_node_indices.append(val)
for dissim_list in U_b_dissimilar.values():
    for u in dissim_list:
        val = user_map.get(u, [])
        if isinstance(val, list):
            expanded_node_indices.extend(val)
        elif val:
            expanded_node_indices.append(val)
expanded_node_indices = list(set(expanded_node_indices))
expanded_node_indices_int = [id_mapping[str(x)] for x in expanded_node_indices if str(x) in id_mapping]

In [11]:
print(len(expanded_node_indices_int))

3307


In [7]:
from tqdm.auto import tqdm
lis = []
node_indices = []  
epoch_iter = tqdm(data_loader)
for batch in epoch_iter:
    batch_g, targets, batch_lbls, node_idx = batch
    
    # Convert node_idx to set for faster lookup
    batch_target_mask = torch.tensor([idx.item() in expanded_node_indices_int for idx in node_idx[targets]])
    
    # Skip if no target nodes in this batch
    if not batch_target_mask.any():
        continue
        
    # Process only if we have target nodes in this batch
    batch_g = batch_g.to(device)
    x = batch_g.ndata.pop("feat")
    prediction = model(batch_g, x)
    batch_emb = prediction[targets][batch_target_mask]

    # Store embeddings and corresponding node indices
    lis.append(batch_emb)
    node_indices.extend(node_idx[targets][batch_target_mask].tolist())

    del prediction, x, batch_g, batch_lbls, batch_target_mask
    torch.cuda.empty_cache()

expanded_embeddings = torch.cat(lis, dim=0)
del lis

100%|██████████| 245700/245700 [22:45<00:00, 179.95it/s] 


In [8]:
# # Create inverse id_mapping (node index -> post ID)
inverse_id_mapping = {v: k for k, v in id_mapping.items()}

user_ids_batch = []
valid_embeddings = []
post_id_to_user = {}
for u, post_ids in user_map.items():
    for p_id in post_ids:
        post_id_to_user[p_id] = u
for i, node_id in enumerate(node_indices):
    post_id = inverse_id_mapping.get(node_id)  # get post_id from node_id
    if post_id is None:
        continue
    found_uid = post_id_to_user.get(post_id)
    if found_uid is None:
        continue
    user_ids_batch.append(found_uid)
    valid_embeddings.append(expanded_embeddings[i])

Number of entries in user_ids_batch: 3307
Sample of user_ids_batch: ['1472502049', '3002409397', '1710865465', '1096787860886298624', '483057195', '1213470305223491584', '38238077', '1553255852', '22142339', '22142339']
Number of embeddings in valid_embeddings: 3307
=== Seed Check ===
Total seeds: 1100
Seeds found in user_ids_batch: 3307
Missing seeds: 0


In [14]:
L_user

tensor(98.7972, device='cuda:0')

In [20]:
L_user, L_sim_dissim

(tensor(98.7972, device='cuda:0', grad_fn=<DivBackward0>),
 tensor(0.2051, device='cuda:0', grad_fn=<DivBackward0>))

In [18]:
# L_sim_dissim = compute_sim_dissim_loss(
#     expanded_embeddings_, user_ids_batch, U_b, U_b_similar, U_b_dissimilar, margin=0.2
# )

import torch
import torch.nn.functional as F

def user_level_contrast_loss_chunked(embeddings, user_ids, tau=0.07, chunk_size=1024):
    device = embeddings.device
    N = embeddings.shape[0]
    if N == 0:
        return torch.tensor(0.0, device=device, requires_grad=True)

    # Convert user_ids to tensor
    if not torch.is_tensor(user_ids):
        # Convert string IDs to integers
        user_ids = [int(uid) for uid in user_ids]  # Parse strings to integers
        user_ids = torch.tensor(user_ids, dtype=torch.long, device=device)

    normed_embeddings = F.normalize(embeddings, dim=-1)
    user_map_index = defaultdict(list)
    for idx, uid in enumerate(user_ids):
        user_map_index[uid.item()].append(idx)

    total_loss = 0.0
    valid_anchors = 0

    for i_start in range(0, N, chunk_size):
        i_end = min(N, i_start + chunk_size)
        anchor_batch = normed_embeddings[i_start:i_end]
        sims = anchor_batch @ normed_embeddings.T
        log_probs = F.log_softmax(sims / tau, dim=1)

        for local_idx, anchor_idx in enumerate(range(i_start, i_end)):
            uid = user_ids[anchor_idx].item()
            pos_indices = [p for p in user_map_index[uid] if p != anchor_idx]
            if len(pos_indices) == 0:
                continue
            pos_log_prob_sum = log_probs[local_idx, pos_indices].sum()
            total_loss += pos_log_prob_sum
            valid_anchors += 1

    if valid_anchors == 0:
        return torch.tensor(0.0, device=device, requires_grad=True)
    return -(total_loss / valid_anchors)
expanded_embeddings_ = torch.stack(valid_embeddings)
L_user = user_level_contrast_loss_chunked(expanded_embeddings_, user_ids_batch, tau=0.07)
L_user

user_ids type: <class 'list'>
user_ids sample: ['1472502049', '3002409397', '1710865465', '1096787860886298624', '483057195']


tensor(216.8539, device='cuda:0', grad_fn=<NegBackward0>)

In [21]:
L_user, L_sim_dissim

(tensor(216.8539, device='cuda:0', grad_fn=<NegBackward0>),
 tensor(0.5768, device='cuda:0', grad_fn=<AddBackward0>))

In [28]:
L_user, L_sim_dissim, L_cluster

(tensor(98.7972, device='cuda:0', grad_fn=<DivBackward0>),
 tensor(0.2051, device='cuda:0', grad_fn=<DivBackward0>),
 tensor(-9.3681, device='cuda:0', grad_fn=<SubBackward0>))

In [20]:
import torch
import torch.nn.functional as F

def compute_sim_dissim_loss(
    embeddings,
    user_ids_batch,
    seed_users,
    U_b_similar,
    U_b_dissimilar,
    margin=0.5,  # Increased margin
    alpha=1.0    # Weight for dissimilar push
):
    """
    Combined triplet and contrastive loss:
    - Pull seed users and similar users closer (contrastive pull).
    - Push seed users and dissimilar users apart (contrastive push).
    - Enforce triplet margin for ranking.

    L_sd = L_triplet + alpha * L_contrastive

    Args:
        embeddings: Tensor [N, d]
        user_ids_batch: List of length N, user_id for each row in 'embeddings'
        seed_users: List U_b (seed)
        U_b_similar: Dict of similar user IDs for each seed user
        U_b_dissimilar: Dict of dissimilar user IDs for each seed user
        margin: Margin for triplet loss
        alpha: Weight for contrastive push term
    """
    device = embeddings.device
    embeddings = F.normalize(embeddings, p=2, dim=-1)  # Normalize embeddings

    user2idx = {uid: i for i, uid in enumerate(user_ids_batch)}
    print(user2idx)
    total_triplet_cost = torch.tensor(0.0, device=device)
    total_contrastive_cost = torch.tensor(0.0, device=device)
    triplet_count = 0
    contrastive_count = 0
    
    seed_users = list(map(str, seed_users))
    for seed_u in seed_users:
        if seed_u not in user2idx:
            continue
        u_idx = user2idx[seed_u]
        z_u = embeddings[u_idx]

        # Similar users (pull closer)
        sim_list = U_b_similar.get(seed_u, [])
        for sim_u in sim_list:
            if sim_u not in user2idx:
                continue
            z_sim = embeddings[user2idx[sim_u]]
            cos_sim = F.cosine_similarity(z_u.unsqueeze(0), z_sim.unsqueeze(0), dim=-1)[0]
            # Contrastive pull: minimize distance (1 - cos_sim)
            contrastive_pull = 1 - cos_sim
            total_contrastive_cost += contrastive_pull
            contrastive_count += 1

            # Dissimilar users (push apart + triplet)
            dissim_list = U_b_dissimilar.get(seed_u, [])
            for dis_u in dissim_list:
                if dis_u not in user2idx:
                    print(dis_u)
                    continue
                z_dis = embeddings[user2idx[dis_u]]
                cos_dis = F.cosine_similarity(z_u.unsqueeze(0), z_dis.unsqueeze(0), dim=-1)[0]
                # Triplet loss
                triplet_cost = torch.relu(margin - (cos_sim - cos_dis))
                total_triplet_cost += triplet_cost
                triplet_count += 1
                # Contrastive push: maximize distance (penalize high cos_dis)
                contrastive_push = torch.relu(cos_dis)  # Push cos_dis toward 0 or negative
                total_contrastive_cost += contrastive_push
                contrastive_count += 1

    if triplet_count == 0:
        print("No valid triplets found")
        return torch.tensor(0.0, device=device)

    triplet_loss = total_triplet_cost / triplet_count
    contrastive_loss = total_contrastive_cost / contrastive_count if contrastive_count > 0 else 0.0
    final_loss = triplet_loss + alpha * contrastive_loss
    
    print(f"Triplet loss: {triplet_loss.item():.4f}, Contrastive loss: {contrastive_loss.item():.4f}, Final loss: {final_loss.item():.4f}")
    return final_loss

L_sim_dissim = compute_sim_dissim_loss(
    embeddings=expanded_embeddings,
    user_ids_batch=user_ids_batch,
    seed_users=U_b,
    U_b_similar=U_b_similar,
    U_b_dissimilar=U_b_dissimilar,
    margin=0.2
)

{'1472502049': 2472, '3002409397': 2513, '1710865465': 2, '1096787860886298624': 20, '483057195': 4, '1213470305223491584': 5, '38238077': 1894, '1553255852': 7, '22142339': 2004, '551250881': 12, '860663394': 13, '109829810': 1925, '177400283': 16, '42054044': 19, '13139632': 3213, '279722486': 22, '1107502425731035136': 2838, '1335852799637159936': 24, '19092695': 25, '1040755878658490368': 26, '21973047': 784, '719842284821422080': 34, '2923479732': 1669, '1467288758': 35, '3036740306': 3174, '1095839282017853440': 2122, '1228438411352166400': 40, '27066982': 41, '2655030456': 44, '318343511': 45, '57042327': 760, '1017789868515188736': 2927, '23368042': 48, '49668631': 2913, '821909367314259968': 1848, '1069693483': 1364, '1918574360': 55, '41172223': 3091, '88430883': 1739, '2302265544': 3236, '1107681972460109824': 3103, '86564120': 3150, '78991193': 88, '15910202': 2944, '588953606': 1428, '3258196377': 946, '956532785748979712': 2588, '3400731045': 2048, '376445018': 2007, '280

In [31]:
def compute_cluster_loss(embeddings, cluster_labels, alpha=1.0):
    """
    Minimizes average distance for same-cluster pairs,
    Maximizes average distance for different-cluster pairs
    via L = mean_intra - alpha * mean_inter
    so that minimizing L => smaller same-cluster dist, bigger diff-cluster dist.
    """
    device = embeddings.device
    N = embeddings.shape[0]
    if N == 0:
        return torch.tensor(0.0, device=device)

    cluster_labels = torch.tensor(cluster_labels, dtype=torch.long, device=device)

    # pairwise dist
    diff = embeddings.unsqueeze(1) - embeddings.unsqueeze(0)  # [N, N, d]
    dist_matrix = (diff * diff).sum(dim=-1)  # [N, N]

    same_cluster_mask = (cluster_labels.unsqueeze(1) == cluster_labels.unsqueeze(0))
    exclude_self = ~torch.eye(N, dtype=torch.bool, device=device)
    same_mask = same_cluster_mask & exclude_self
    diff_mask = ~same_cluster_mask & exclude_self

    same_dist = dist_matrix[same_mask]
    diff_dist = dist_matrix[diff_mask]

    if same_dist.numel() > 0:
        mean_intra = same_dist.mean()
    else:
        mean_intra = torch.tensor(0.0, device=device)

    if diff_dist.numel() > 0:
        mean_inter = diff_dist.mean()
    else:
        mean_inter = torch.tensor(0.0, device=device)

    # L = mean_intra - alpha * mean_inter
    # Minimizing => reduce same-cluster distance, increase diff-cluster distance
    loss = mean_intra - alpha * mean_inter
    return loss
L_cluster = compute_cluster_loss(expanded_embeddings, 5, alpha=1.0)
L_cluster

IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)

In [19]:
model = model.to(device)
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
# total_loss = L_user + L_sim_dissim + 0.7 * L_cluster
total_loss = L_user
optimizer.zero_grad()
total_loss.backward()
optimizer.step()